##Spark Aggregate Functions

###Funciones simple de agregacion

In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movies_df.show(4)

### Funcion Count

In [0]:
#Count
movies_df.select(count("*")).show()

In [0]:
#Vemos quie hay 299 registros nulos para el campo "year_release_date"
movies_df.filter(
                  col("year_release_date").isNull()
                ).count()

#entonces, si contamos por ese campo, salen 4426 registros no nulos
movies_df.select(count("year_release_date")).display()

#Con countDistinct, cuenta valores diferentes
movies_df.select(countDistinct("year_release_date")).display()


### SUM

In [0]:
movies_df.select(sum("budget")).display()

In [0]:

movies_df.filter(
                (col("year_release_date") == 2016)
               )\
         .select(sum("budget"), count("movie_id"))\
         .withColumnRenamed("sum(budget)", "total_budget")\
         .withColumnRenamed("count(movie_id)", "count_movies")\
         .display()


##Group BU

In [0]:
movies_df.groupBy("year_release_date")\
         .sum("budget")\
         .display()

movies_df.groupBy("year_release_date")\
         .agg(sum("budget").alias("sum_budget"))\
         .display()


In [0]:
movies_df.display()

In [0]:
movies_df.filter(
                 (col("year_release_date").isNotNull())
                )\
          .groupby("year_release_date", "title")\
          .agg(
                count("movie_id").alias("count_movies"),
                sum("budget").alias("sum_budget"),
                max("budget").alias("max_budget")
              )\
          .display()        

##Windows Functions


In [0]:
from pyspark.sql.functions import rank, dense_rank, desc
from pyspark.sql.window import Window

In [0]:
movies_df.select("title","budget", "year_release_date")

In [0]:
movies_df.select("title","budget", "year_release_date")\
          .filter(
                   col("year_release_date").isNotNull() &
                   (col("year_release_date") == 1960)
                 ) \
          .withColumn("dense_rank", dense_rank().over(Window.partitionBy("year_release_date")
                                               .orderBy(desc("budget"))
                                               #.orderBy(desc("title"))
                                         )
                     )\
          .withColumn("rank", rank().over(Window.partitionBy("year_release_date")
                                               .orderBy(desc("budget"))
                                               #.orderBy(desc("title"))
                                         )
                     )\
         .display()
